# Social coordination task {#sec-socialcoord}

In this section, we investigate the impact of optimism/pessimism in a social coordination task. To do this, we use the stag-hunt game defined in @sec-socialdilemma . Additionally, we study the effect of optimism/pessimism on cooperation in a prisoner's dilemma. 

In previous studies (such as [@matignon2007]), it has been suggested that optimism enhances cooperation on coordination tasks. This is what we replicate in this section. 

First, we import what we need:

In [ ]:
# general imports

import numpy as np
from scipy.stats import norm

import jax.numpy as jnp
from jax import jit
from functools import partial

import matplotlib.pyplot as plt
import matplotlib as mpl

import pandas as pd
import seaborn as sns

In [ ]:
# pyCRLD imports

# Agents
from pyCRLD.Agents.Base import abase
from pyCRLD.Agents.StrategySARSA import stratSARSA

# Environment
from pyDDRL.Environments.SocialDilemma import SocialDilemma

# Utilities
from pyCRLD.Utils.Helpers import *
from pyDDRL.Utils.Trajectories1D import trajectories_1D 
from pyDDRL.Utils.PhasePlot import phaseplot, _data_to_plot, _strategies, _dXisa_s
from pyDDRL.Utils.PlottingTools import *

In [ ]:
# DDRL imports

from pyDDRL.Agents.DDRLBase import DDRLBase
from pyDDRL.Agents.DDRLSarsa import DDRLSarsa

Then, we define the environment:

In [ ]:
env = SocialDilemma(R=3, P=0, S=-2, T=1)

## Sanity tests

We carry out sanity tests in order to check that the behavior of neutral DDRLSarsa agents imitate the behavior of regular SARSA agents. Once again, we compare the "mean", "expectile", and "weights" methods. 

In [ ]:
# Methods
methods = ["mean", "expectile", "weights"]

# Define agents to compare
aeis_ddrl = [] # DDRLSarsa agents
aeis_reg = []  # regular SARSA agents

for i, method in enumerate(methods):
    aei_ddrl = DDRLSarsa(env, nr_reward_bins=29, discretization_sigma=0.08, 
                        learning_rates=0.1, discount_factors=0.9, 
                        choice_intensities=50, Rmin=-2, Rmax=3,
                        method=method, use_prefactor=True)
    aeis_ddrl.append(aei_ddrl)
    
    aei_reg = stratSARSA(env=env, learning_rates=0.1, discount_factors=0.9, 
                         choice_intensities=50, use_prefactor=True)
    aeis_reg.append(aei_reg)

Additionally, we plot trajectories with various initial conditions for better visualization.

In [ ]:
#| label: fig-socialcoord-sanitytests
#| fig-cap: "Comparison between a distributional and a non-distributional agent on a stag-hunt game. We compare distributional agents with a weights method ($w_U = 1$), with an expectile method ($\tau=0.5$), and taking the mean of the Q-return distribution."

# Phase space items to plot along the axes
x = ([0], [0], [0])  # agent 1
y = ([1], [0], [0])  # agent 2

# For trajectories
Xinits = []
xys = [(0.5, 0.5), (0.295, 0.205), (0.205, 0.695), (0.805, 0.305), (0.705, 0.795)]
cols = ['orange', 'purple', 'green', 'pink', 'brown']
        
for t, xy in enumerate(xys):
    Xinit = np.ones((2, 1, 2)) * 0.5
    Xinit[0, 0, 0] = xy[0]
    Xinit[1, 0, 0] = xy[1]
    Xinit[0, 0, 1] = 1 - Xinit[0, 0, 0]
    Xinit[1, 0, 1] = 1 - Xinit[1, 0, 0]
    Xinits.append(Xinit)

# Decorations
titles = ["method = mean", "method = expectile", "method = weights"]
xlabel = "prob. agent 1 cooperating"
ylabel = "prob. agent 2 cooperating"

# Figure and axes
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(3.2 * 3.8, 3.8))
axes = [ax1, ax2, ax3]

# Plot flow plots and trajectories
plot_3flowplots(aeis_reg, x, y, NrRandom=16, cmap='Reds', axes=axes)
plot_3flowplots(aeis_ddrl, x, y, NrRandom=16, cmap='Blues', decorate=True,
                titles=titles, xlabel=xlabel, ylabel=ylabel,
                Xinits=Xinits, cols=cols, axes=axes)

# Additional decorations
ax3.plot([], [], color='red', label='detSarsa')
ax3.plot([], [], color='blue', label='DDRLSarsa')
ax3.legend(bbox_to_anchor=(1.01, 1.01))

# Save the figure
plt.savefig('../figs/SHSanityTests.eps', dpi=300, bbox_inches='tight', format='eps')

plt.show()

<Figure size 3648x1140 with 3 Axes>

## Influence of optimism/pessimism

Once again, neutral weights-DDRLSarsa agents show less errors than neutral expectile-DDRLSarsa agents. Therefore, we use the weights method to explore the effects of optimism and pessimism on cooperation in the stag-hunt game. In particular, we look at how optimism/pessimism impacts the separatrix of the flowplot. This separatrix demarcates the domain where all initial strategies lead to mutual cooperation from the domain where all initial strategies lead to mutual defection. 

We compare a pessimistic agent with $w_U=0.25$, with a neutral agent ($w_U=1$) and an optimistic agent ($w_U = 4$).

In [ ]:
weights = [0.25, 1., 4.]

# Define agents to compare
aeis = [] # DDRLSarsa agents

for i, weight in enumerate(weights):
    aei = DDRLSarsa(env, nr_reward_bins=29, discretization_sigma=0.08, 
                    learning_rates=0.1, discount_factors=0.9, 
                    choice_intensities=50, Rmin=-2, Rmax=3,
                    method="weights", wU=weight, use_prefactor=True)
    aeis.append(aei)

Now plotting:

In [ ]:
#| label: fig-sh-flowplots
#| fig-cap: "Flow plots on a stag-hunt game for a pessimistic, neutral and optimistic agent."

# Decorations
titles = [r'$w_U=0.25$', r'$w_U=1$', r'$w_U=4$']
xlabel = "prob. agent 1 cooperating"
ylabel = "prob. agent 2 cooperating"

# Figure and axes
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(3.2 * 3.8, 3.8))
axes = [ax1, ax2, ax3]

# Plot flow plots and trajectories
plot_3flowplots(aeis, x, y, NrRandom=16, cmap='Blues', decorate=True,
                titles=titles, xlabel=xlabel, ylabel=ylabel,
                Xinits=Xinits, cols=cols, axes=axes)

# Plot separatrices
for i, aei in enumerate(aeis):
    ax = axes[i]
    plot_separatrix(aei, ax, x, y)

# Save the figure
plt.savefig('../figs/SHFlowPlots.eps', dpi=300, bbox_inches='tight', format='eps')

plt.show()

<Figure size 3648x1140 with 3 Axes>

We can also plot the phase plot for one agent, assuming the other agent always plays the same strategy as the first one. This is equivalent with visualizing the diagonal of the above flowplots. 

In [ ]:
#| label: fig-sh-phaseplot
#| fig-cap: "Phase plot, strategy plot and reward plot of an agent playing a stag-hunt game. Different levels of optimism/pessimism are compared."

weights = np.array([0.1, 0.2, 0.4, 0.8, 1., 2., 4., 8., 10.])

x = ([0], [0], [0]) 
res = np.linspace(0, 1., 21)
res[0] = 0.0001
res[-1] = 0.9999
otherXs = np.linspace(0, 1., 21)
otherXs[0] = 0.0001
otherXs[-1] = 0.9999

compare_levels(env, weights, gamma=0.9, beta=50., Rmin=-2, Rmax=3, 
               xinds=x, res=res, nrb=29, discsig=0.08, traj_len=30, 
               otherXs=otherXs, cmap='coolwarm', save=True, 
               savename='../figs/SHPhasePlots.eps')

plt.show()

<Figure size 5130x1140 with 3 Axes>

## Optimism has no effect in a prisoner's dilemma

As a complement, we can turn the stag-hunt game into a prisoner's dilemma, which is another social dilemma with a different reward structure. In a prisoner's dilemma, typically, mutual cooperation is not an equilibrium; both agents are incentivized to defect. First, we define the environment:

In [ ]:
env = SocialDilemma(R=1, P=0, S=-2, T=3)

And we carry the same sanity tests as above:

In [ ]:
methods = ["mean", "expectile", "weights"]

# Define agents to compare
aeis_ddrl = [] # DDRLSarsa agents
aeis_reg = []  # regular SARSA agents

for i, method in enumerate(methods):
    aei_ddrl = DDRLSarsa(env, nr_reward_bins=29, discretization_sigma=0.08, 
                         learning_rates=0.1, discount_factors=0.9, 
                         choice_intensities=50, Rmin=-2, Rmax=3,
                         method=method, use_prefactor=True)
    aeis_ddrl.append(aei_ddrl)
    
    aei_reg = stratSARSA(env=env, learning_rates=0.1, discount_factors=0.9, 
                         choice_intensities=50, use_prefactor=True)
    aeis_reg.append(aei_reg)

In [ ]:
#| label: fig-pd-sanitytests
#| fig-cap: "Comparison between a distributional and a non-distributional agent on a prisoner's dilemma. We compare distributional agents with a weights method, with an expectile method, and taking the mean of the Q-return distribution."

# Phase space items to plot on axes
x = ([0], [0], [0])  # agent 1
y = ([1], [0], [0])  # agent 2

# For trajectories
Xinits = []
xys = [(0.5, 0.5), (0.295, 0.205), (0.205, 0.695), (0.805, 0.305), (0.705, 0.795)]
cols = ['orange', 'purple', 'green', 'pink', 'brown']
        
for t, xy in enumerate(xys):
    Xinit = np.ones((2, 1, 2)) * 0.5
    Xinit[0, 0, 0] = xy[0]
    Xinit[1, 0, 0] = xy[1]
    Xinit[0, 0, 1] = 1 - Xinit[0, 0, 0]
    Xinit[1, 0, 1] = 1 - Xinit[1, 0, 0]
    Xinits.append(Xinit)

# Decorations
titles = ["method = mean", "method = expectile", "method = weights"]
xlabel = "prob. agent 1 cooperating"
ylabel = "prob. agent 2 cooperating"

# Figure and axes
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(3.2 * 3.8, 3.8))
axes = [ax1, ax2, ax3]

# Plot flow plots and trajectories
plot_3flowplots(aeis_reg, x, y, NrRandom=16, cmap='Reds', axes=axes)
plot_3flowplots(aeis_ddrl, x, y, NrRandom=16, cmap='Blues', decorate=True,
                titles=titles, xlabel=xlabel, ylabel=ylabel,
                Xinits=Xinits, cols=cols, axes=axes)

# Additional decorations
ax3.plot([], [], color='red', label='detSarsa')
ax3.plot([], [], color='blue', label='DDRLSarsa')
ax3.legend(bbox_to_anchor=(1.01, 1.01))

# Save the figure
plt.savefig('../figs/PDSanityTests.eps', dpi=300, bbox_inches='tight', format='eps')

plt.show()

<Figure size 3648x1140 with 3 Axes>

We can investigate the influence of optimism or pessimism by plotting the same phase plots and strategy plots as above, assuming the second agent always behaves like the first. 

In [ ]:
weights = np.array([0.1, 0.2, 0.4, 0.8, 1., 2., 4., 8., 10.])

x = ([0], [0], [0]) 
res = np.linspace(0, 1., 21)
res[0] = 0.0001
res[-1] = 0.9999
otherXs = np.linspace(0, 1., 21)
otherXs[0] = 0.0001
otherXs[-1] = 0.9999

We can prove that optimism has an effect on cooperation in a stag-hunt game, but not in a prisoner's dilemma.

## Proof: Effect of optimism on social dilemmas

We consider a social dilemma, in which two agents can cooperate ($c$) or defect ($d$).

Mutual cooperation yields a "reward" $R$ for both agents, while mutual defection yields a "punishment" $P$. If one agent cooperates and the other defects, the cooperator gets a "sucker" payoff $S$, while the defector gets a "temptation" payoff $T$. The stag-hunt game and the prisoner's dilemma differ in their reward structure:

* In a stag-hunt game, $R>T>P>S$;
* In a prisoner's dilemma, $T>R>P>S$.

We want to find the conditions under which the Q-value of cooperation for agent $i$, $Q^{i,c}$, is greater than the Q-value for defection for agent $i$, $Q^{i,d}$. To do this, we consider a weights-DDRL agent with number of neurons $n$, discretization sigma $\sigma$, discretization grid ${x_j, j \in [1, n]}$ with width $\Delta x$, and weight vector $\boldsymbol{w}$ with elements $w_j$, $j \in [1, n]$. Moreover, we place ourselves in the limit of an infinitely small discretization sigma, $\sigma \rightarrow 0$, and we assume the agent's discount factor $\gamma$ is $0$. As a consequence, the Q-return distributions $Q^{i,c,v}$ and $Q^{i,d,v}$ just depend on the payoff and on the other agent $j$'s strategy $X^{j,c}$.

* $Q^{i,c,v} = R$ with probability $X^{j,c}$;
* $Q^{i,c,v} = S$ with probability $1 - X^{j,c}$;
* $Q^{i,d,v} = P$ with probability $X^{j,d} = 1 - X^{j,c}$;
* $Q^{i,d,v} = T$ with probability $1 - X^{j,d} = X^{j,c}$.

The mean Q-value $\bar Q^{i,c,v}$ is therefore $X^{j,c} \cdot R + (1 - X^{j,c}) \cdot S = S + X^{j,c} (R - S)$. We consider $X^{j,c} \in ]0,1[$. In both social dilemmas, $R > S$, from which it follows that $S < \bar Q^{i,c,v} < R$. As a consequence, the biased and normalized distribution $\tilde Q^{i,c,v}$ is described by:

* $Q^{i,c,v} = R$ with probability $\frac{w_U \cdot X^{j,c}}{w_U \cdot X^{j,c} + (1 - X^{j,c})}$;
* $Q^{i,c,v} = S$ with probability $\frac{1 - X^{j,c}}{w_U \cdot X^{j,c} + (1 - X^{j,c})}$.

A similar reasoning can be applied to the Q-return distribution $Q^{i,d,v}$, and we obtain:

* $Q^{i,d,v} = P$ with probability $\frac{1 - X^{j,c}}{w_U \cdot X^{j,c} + (1 - X^{j,c})}$;
* $Q^{i,d,v} = T$ with probability $\frac{w_U \cdot X^{j,c}}{w_U \cdot X^{j,c} + (1 - X^{j,c})}$.

Consequently, the means of these biased distributions are:

\begin{equation}
    \overset{\simeq} Q^{i,c,v} = \frac{w_U \cdot X^{j,c} \cdot R + (1 - X^{j,c})\cdot S}{w_U \cdot X^{j,c} + (1 - X^{j,c})}\\
    = \frac{X^{j,c} \cdot (w_U \cdot R - S) + S}{X^{j,c} \cdot (w_U - 1) + 1}
\end{equation}

\begin{equation}
    \overset{\simeq} Q^{i,d,v} = \frac{w_U \cdot X^{j,c} \cdot T + (1 - X^{j,c})\cdot P}{w_U \cdot X^{j,c} + (1 - X^{j,c})}\\
    = \frac{X^{j,c} \cdot (w_U \cdot T - P) + P}{X^{j,c} \cdot (w_U - 1) + 1}
\end{equation}

We have $\overset{\simeq} Q^{i,c,v} \geq \overset{\simeq} Q^{i,c,d}$

iff $X^{j,c} \cdot (w_U \cdot R - S) + S \geq X^{j,c} \cdot (w_U \cdot T - P) + P$

iff $X^{j,c} \cdot [w_U (R-T) + (P-S)] + (S-P) \geq 0$

iff $X^{j,c} \cdot [w_U (R-T)] + (S-P)[1 - X^{j,c}] \geq 0$.

In both social dilemmas, $P > S$ so $S-P < 0$. Therefore, $(S-P)[1 - X^{j,c}] < 0$.

In a prisoner's dilemma, $T > R$, so $R-T < 0$, and as a consequence, $X^{j,c} \cdot [w_U (R-T)] < 0$. Therefore, in a prisoner's dilemma, it is not possible to have $\overset{\simeq} Q^{i,c,v} \geq \overset{\simeq} Q^{i,c,d}$. In other terms, the agents always mutually defect.

In a stag-hunt game, however, $R > T$, so $R-T > 0$, and as a consequence, $X^{j,c} \cdot [w_U (R-T)] > 0$. So we have $\overset{\simeq} Q^{i,c,v} \geq \overset{\simeq} Q^{i,c,d}$

iff $w_U (R-T) \geq \frac{(P-S)(1 - X^{j,c})}{X^{j,c}}$

iff $w_U \geq \frac{(P-S)(1 - X^{j,c})}{(R-T)X^{j,c}}$.

Therefore, in a stag-hunt game, optimism/pessimism has an effect on cooperation. 

We can plot $w_U$ as a function of $X^{j,c}$ for the stag-hunt game we investigate in our paper, that is: $R=3$, $T=1$, $P=0$, and $S=-2$.

In [ ]:
#| label: fig-sh-phaseplot-pd
#| fig-cap: "Phase plot, strategy plot and reward plot of an agent playing a prisoner's dilemma. Different levels of optimism/pessimism are compared."

compare_levels(env, weights, gamma=0.9, beta=50., Rmin=-2, Rmax=3, 
               xinds=x, res=res, nrb=29, discsig=0.08, traj_len=30, 
               otherXs=otherXs, cmap='coolwarm', save=True, 
               savename='figs/PDPhasePlots.eps')

plt.show()

<Figure size 5130x1140 with 3 Axes>

In [ ]:
def w_coop(X, R=3, T=1, P=0, S=-2):
    return (P-S) * (1-X) / ((R-T) * X)

In [ ]:
X = np.linspace(0.05, 1, 20)
ws = w_coop(X)
ones = np.ones((len(X)))

plt.plot(X, ws, 'o-')
plt.plot(X, ones, '--', color="gray", label=r'$w_U = 1$')
plt.xlabel("probability of agent 2 cooperating")
plt.ylabel("minimum "+r'$w_U$'+" for agent 1 to cooperate")
plt.legend()

plt.show()

<Figure size 2700x2100 with 1 Axes>